### IMPORTING THE NECESSARY LIBRARIES AND STUFF

In [1]:
import os
import shutil
import yaml

### FINDING OUT THE PATHS TO THE INPUT FILES (CONTAINING THE OUTPUT OF THE LOCALLY RUN .py SCRIPTS)

In [2]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/csdd.yaml
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/hyp.simulation16.yaml
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/hyp.simulation32.yaml
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/hyp.simulation43.yaml
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/hyp.simulation.yaml
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/hyp.simulation5.yaml
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/hyp.simulation25.yaml
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/labels/val/0389.txt
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/labels/val/1356.txt
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/labels/val/0534.txt
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/labels/val/0333.txt
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/labels/val/1038.txt
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/labels/val/0069.txt
/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws/labels/val/2013.txt
/kaggle/input/datasets/arnavde/csdd

### MOVING THE INPUT FILES TO OUTPUT (/kaggle/working) AS INPUT FILES CANNOT BE MODIFIED IN KAGGLE

In [3]:
SOURCE = '/kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws' 
DEST = '/kaggle/working/ws'

if os.path.exists(SOURCE):
    if os.path.exists(DEST):
        shutil.rmtree(DEST)
    
    shutil.copytree(SOURCE, DEST)
    print(f" Data copied from {SOURCE} to {DEST}")
else:
    print(" Source folder not found.")

 Data copied from /kaggle/input/datasets/arnavde/csdd-ws1x/CSDD_ws to /kaggle/working/ws


### UPDATING PATHS IN THE .yaml FILE AS THEY ORIGINALLY WRITTEN TO BE RUN LOCALLY

In [4]:
yaml_path = '/kaggle/working/ws/csdd.yaml'
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

config['path'] = '/kaggle/working/ws' 
config['train'] = 'img/train'
config['val'] = 'img/val'
config['test'] = 'img/test'

with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(" csdd.yaml paths updated to /kaggle/working/ws/")

 csdd.yaml paths updated to /kaggle/working/ws/


### INSTALLING YOLOV5 

In [5]:
if not os.path.exists('yolov5'):
    print("Cloning YOLOv5 repository...")
    !git clone https://github.com/ultralytics/yolov5
    %cd yolov5
    print("Installing dependencies...")
    !pip install -r requirements.txt -q
    !pip install albumentations==1.4.11 --force-reinstall # to use the blur and other albumentations so that image detection works better in the hazy camera feed of gazebo
else:
    %cd yolov5
    print("YOLOv5 environment already initialized.")

print("Environment Ready: YOLOv5 and all dependencies are active.")

Cloning YOLOv5 repository...
Cloning into 'yolov5'...
remote: Enumerating objects: 17898, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 17898 (delta 39), reused 9 (delta 9), pack-reused 17850 (from 5)
Receiving objects: 100% (17898/17898), 17.06 MiB | 27.47 MiB/s, done.
Resolving deltas: 100% (12193/12193), done.
/kaggle/working/yolov5
Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1

### YOLOV5 EXPECTS FOLDER WITH THE NAME "images" AND NOT "img"; AND CONSEQUENTLY UPDATING THE csdd.yaml FILE

In [6]:
if os.path.exists('/kaggle/working/ws/img'):
    os.rename('/kaggle/working/ws/img', '/kaggle/working/ws/images')
    print(" Folder 'img' renamed to 'images'")

yaml_path = '/kaggle/working/ws/csdd.yaml'
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

config['train'] = 'images/train'
config['val'] = 'images/val'
config['test'] = 'images/test'

with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(" csdd.yaml updated with 'images' paths.")

 Folder 'img' renamed to 'images'
 csdd.yaml updated with 'images' paths.


### MOVING THE csdd.yaml AND hyp.simulation.yaml INTO THE yolov5 FOLDER IN PROPER PLACES WHERE IT CAN DETECT

In [7]:
source = '/kaggle/working/ws/csdd.yaml'
destination = '/kaggle/working/yolov5/data'

os.makedirs(os.path.dirname(destination), exist_ok=True)

try:
    shutil.move(source, destination)
    print("File moved successfully!")
except FileNotFoundError:
    print("Error: The source file was not found.")
except PermissionError:
    print("Error: Permission denied.")

File moved successfully!


In [8]:
source = '/kaggle/working/ws/hyp.simulation43.yaml'
destination = '/kaggle/working/yolov5/data/hyps'

os.makedirs(os.path.dirname(destination), exist_ok=True)

try:
    shutil.move(source, destination)
    print("File moved successfully!")
except FileNotFoundError:
    print("Error: The source file was not found.")
except PermissionError:
    print("Error: Permission denied.")

File moved successfully!


In [9]:
%cd /kaggle/working/yolov5

%env PYTHONWARNINGS=ignore

!python -m torch.distributed.run --nproc_per_node 2 train.py \
                 --img 1024 \
                 --batch 32 \
                 --epochs 100 \
                 --data data/csdd.yaml \
                 --weights yolov5s.pt \
                 --device 0,1 \
                 --optimizer SGD \
                 --hyp data/hyps/hyp.simulation43.yaml \
                 --name CSDD_Model \
                 --exist-ok | grep -E "Epoch|mAP|Class" # Ensures output of 1 line per epoch to prevent flooding of output

/kaggle/working/yolov5
env: PYTHONWARNINGS=ignore

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-04-22 16:45:40.433379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-22 16:45:40.433380: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776876340.658770     101 cuda_dnn.cc:8579] Unable to register

In [10]:
%cd /kaggle/working/yolov5

!python val.py --weights runs/train/CSDD_Model/weights/best.pt \
               --data data/csdd.yaml \
               --img 1024 \
               --task test \
               --name CSDD_Model_Test



/kaggle/working/yolov5
val: data=data/csdd.yaml, weights=['runs/train/CSDD_Model/weights/best.pt'], batch_size=32, imgsz=1024, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=CSDD_Model_Test, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-472-g7ca403a3 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7018216 parameters, 0 gradients, 15.8 GFLOPs
test: Scanning /kaggle/working/ws/labels/test... 422 images, 0 backgrounds, 0 co
test: New cache created: /kaggle/working/ws/labels/test.cache
                 Class     Images  Instances          P          R      mAP50   
                   all        422      11769      0.723      0.614      0.647      0.352
               Scratch        422       6711      0.797      0.777       0.81      0.528
                  

In [11]:
WEIGHTS_PATH = '/kaggle/working/yolov5/runs/train/CSDD_Model/weights/best.pt'
TEST_IMAGES_PATH = '/kaggle/working/ws/images/test'

!python detect.py --weights {WEIGHTS_PATH} \
                  --source {TEST_IMAGES_PATH} \
                  --img 1024 \
                  --conf 0.25 \
                  --name CSDD_Test_Images 

detect: weights=['/kaggle/working/yolov5/runs/train/CSDD_Model/weights/best.pt'], source=/kaggle/working/ws/images/test, data=data/coco128.yaml, imgsz=[1024, 1024], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=CSDD_Test_Images, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-472-g7ca403a3 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7018216 parameters, 0 gradients, 15.8 GFLOPs
image 1/422 /kaggle/working/ws/images/test/0007.jpg: 1024x1024 8 Scratchs, 2 Spots, 22.9ms
image 2/422 /kaggle/working/ws/images/test/0010.jpg: 1024x1024 6 Scratchs, 2 Spots, 22.9ms
image 3/422 /kaggle/working/ws/images/test/0011.jpg: 1024x1024 5 Scratchs, 

### SAVE THE ENTIRE OUTPUT AS A ZIP FILE

In [12]:
%cd /kaggle/working
!zip -r CSDD_Results_IITG.zip yolov5/runs -q

/kaggle/working
